## Planning:
- Load data
- Create sample TE
- Create TE in run.cv
- Implement for XGBoost

In [1]:
import pandas as pd
import numpy as np

In [65]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
submit = pd.read_csv("sample_submission.csv")

In [66]:
# proportion of each class
train['is_fraud'].value_counts(normalize=True)

is_fraud
0.0    0.9833
1.0    0.0167
Name: proportion, dtype: float64

In [67]:
# check correlation of numeric variable
# take all column numeric column
numeric_cols = train.drop(columns='id').select_dtypes(include='number').columns.tolist()

In [68]:
numeric_cols

['amount_usd',
 'hours_since_last_txn',
 'txn_count_last_24h',
 'distance_from_home_km',
 'card_age_months',
 'customer_age',
 'account_balance_usd',
 'cvv_retry_count',
 'velocity_score',
 'time_of_day_hour',
 'day_of_week',
 'merchant_risk_score',
 'prior_disputes',
 'is_fraud']

In [69]:
numeric_cols_ex = numeric_cols.copy()

In [70]:
numeric_cols_ex.remove("is_fraud")

In [71]:
numeric_cols_ex

['amount_usd',
 'hours_since_last_txn',
 'txn_count_last_24h',
 'distance_from_home_km',
 'card_age_months',
 'customer_age',
 'account_balance_usd',
 'cvv_retry_count',
 'velocity_score',
 'time_of_day_hour',
 'day_of_week',
 'merchant_risk_score',
 'prior_disputes']

In [72]:
bool_cols = train.select_dtypes(include='bool').columns.tolist()

In [73]:
obj_cols = train.select_dtypes(include='object').columns.tolist()

In [74]:
obj_cols

['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']

In [75]:
train[obj_cols].head(5)

,merchant_category,card_type,auth_method,channel,device_type
0,Groceries,Mastercard,OTP,POS,iPhone
1,Online Retail,Visa,PIN,In-App,Android Phone
2,Utilities,Visa,OTP,Online,Tablet
3,Groceries,Mastercard,3D Secure,POS,Android Phone
4,Electronics,Visa,3D Secure,Online,Windows PC


In [76]:
for col in obj_cols:
    counts = train.groupby(col)["id"].count()
    proportion = counts/counts.sum()

    print(f'col: {col}')
    summary = pd.DataFrame({'count': counts, 'proportion': proportion})
    print(summary)
    print('-'*20)

col: merchant_category
                    count  proportion
merchant_category                    
Crypto Exchange      5285    0.007550
Electronics         48693    0.069565
Fuel                59880    0.085548
Gaming              23784    0.033979
Gift Cards           9456    0.013509
Groceries          142578    0.203694
Healthcare          35889    0.051273
Online Retail      155097    0.221580
Restaurants        101360    0.144808
Streaming           47196    0.067427
Travel              28715    0.041024
Utilities           42027    0.060042
--------------------
col: card_type
             count  proportion
card_type                     
Amex         45797    0.065428
Discover     17235    0.024623
Mastercard  254857    0.364102
RuPay        37070    0.052960
Visa        345001    0.492887
--------------------
col: auth_method
                    count  proportion
auth_method                          
3D Secure          210036    0.300069
Biometric           93666    0.133816
No

In [77]:
# taking sample data for development
train_sample = train.sample(frac=0.01).reset_index(drop=True)

In [78]:
train_sample.head(3)

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
0,377509,25.91,3.78,3,5.51,55,35,2338.74,0,17.6,...,3D Secure,In-App,POS Terminal,False,False,False,False,False,False,0.0
1,271650,5.43,9.98,2,15.87,48,58,1427.45,0,6.0,...,Biometric,Online,Android Phone,False,False,False,False,False,False,0.0
2,428420,45.31,3.80,4,6.19,45,32,4746.12,0,27.2,...,3D Secure,Online,POS Terminal,False,False,False,False,False,False,0.0


In [79]:
train_sample.to_csv("train_sample.csv",index=False)

In [80]:
test_sample = test.sample(frac=0.01).reset_index(drop=True)

In [81]:
test_sample[obj_cols].head(3)

,merchant_category,card_type,auth_method,channel,device_type
0,Groceries,Mastercard,3D Secure,ATM,Tablet
1,Fuel,Discover,Biometric,POS,iPhone
2,Restaurants,Visa,OTP,Contactless,Smart Watch


## Experimenation and Sample TE Encoding

In [82]:
from sklearn.preprocessing import TargetEncoder

In [83]:
#Splitting the data
y = train_sample['is_fraud']
features = train_sample[obj_cols]
#Specify the target type
te = TargetEncoder(smooth="auto",target_type='binary')
te.fit(features,y)
X_trans = te.transform(features)
#Creating a Dataframe
features_encoded = pd.DataFrame(X_trans, columns = features.columns)

In [90]:
# features_encoded

In [91]:
# transform on test
X_test_te = te.transform(test_sample[obj_cols])

In [92]:
X_test_te

array([[0.00694205, 0.01641849, 0.01574704, 0.02557238, 0.01413328],
       [0.00668361, 0.        , 0.00106376, 0.01224959, 0.01253208],
       [0.01078666, 0.01404408, 0.01060218, 0.01699355, 0.02430153],
       ...,
       [0.01078666, 0.01641849, 0.01574704, 0.01164128, 0.01413328],
       [0.01078666, 0.01404408, 0.00581615, 0.01164128, 0.01506975],
       [0.01031098, 0.00807362, 0.01574704, 0.01699355, 0.01111659]])

In [93]:
X_test_te = pd.DataFrame(X_test_te, columns = obj_cols)

In [94]:
X_test_te.head(3)

,merchant_category,card_type,auth_method,channel,device_type
0,0.006942,0.016418,0.015747,0.025572,0.014133
1,0.006684,0.000000,0.001064,0.012250,0.012532
2,0.010787,0.014044,0.010602,0.016994,0.024302


In [96]:
X_test_te.shape

(3000, 5)

In [97]:
# return the result of encoding to the test_sample
# test_sample[obj_cols] = pd.DataFrame(X_test_te, columns = obj_cols)
test_sample.loc[:,obj_cols] = te.transform(test_sample[obj_cols])

In [98]:
test_sample.head(2)

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,card_type,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt
0,949150,20.43,1.57,3,0.16,40,58,3935.75,0,23.9,...,0.016418,0.015747,0.025572,0.014133,False,True,False,False,False,False
1,817316,16.99,12.23,5,22.87,44,43,1211.63,0,18.3,...,0.0,0.001064,0.01225,0.012532,False,True,False,False,False,False


In [100]:
test_sample.shape

(3000, 25)

In [103]:
# change data type to float64
for col in obj_cols:
    test_sample[col] = test_sample[col].astype('float64')
    # test_sample.loc[:,col] = test_sample.loc[:,col].astype('float64')

In [104]:
test_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 25 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            3000 non-null   int64  
 1   amount_usd                    3000 non-null   float64
 2   hours_since_last_txn          3000 non-null   float64
 3   txn_count_last_24h            3000 non-null   int64  
 4   distance_from_home_km         3000 non-null   float64
 5   card_age_months               3000 non-null   int64  
 6   customer_age                  3000 non-null   int64  
 7   account_balance_usd           3000 non-null   float64
 8   cvv_retry_count               3000 non-null   int64  
 9   velocity_score                3000 non-null   float64
 10  time_of_day_hour              3000 non-null   int64  
 11  day_of_week                   3000 non-null   int64  
 12  merchant_risk_score           3000 non-null   float64
 13  pri

In [30]:
test_sample.head(2)

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,card_type,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt
0,960930,449.93,3.66,1,30.91,40,40,725.69,0,15.2,...,0.014069,0.009931,0.008591,0.011099,False,True,False,False,False,False
1,980156,18.53,6.69,2,4.95,44,45,2449.63,0,25.7,...,0.014069,0.012430,0.016624,0.015605,False,True,False,False,False,False
